# Bottom age tracer in the OM3-25km beta release

The age tracer is set every time step to zero at the surface. Young bottom age tracer values indicate recent contact with the surface, allowing us to use it as a proxy for deep convection, Dense Shelf Water and Antarctic Bottom Water formation, and their pathways.

Many global models struggle to simulate Antarctic Bottom Water formation through Dense Shelf Water export down the continental slope, where it mixes with surrounding waters; instead, they often form it in the open ocean through deep convection.

The bottom age tracer helps investigate how models form Antarctic Bottom Water. Since Dense Shelf Water export is sensitive to stratification, a model's ability to simulate these processes may change during the run. Therefore, we recommend examining the bottom age tracer early in the simulation (e.g., around year 10) and towards the end. The year can be selected in the cell following `Select year to plot`.

In [ ]:
#parameters

# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"
# esm_file = "/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-27-07-2026-PD-control/cm3-datastore/cm3-datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
# Parameters
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"
papermill = True
cwd = "/g/data/v45/aek156/notebooks/github/ACCESS-Community-Hub/access-om3-paper-1/notebooks/mkfigs_output_MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/"
nbname = "Bottom_age_tracer_in_ACCESS_OM3.ipynb"


In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)
from model_agnostic import patch_broken_conda_env
from exptdata_access import get_exptname_from_path
patch_broken_conda_env()  # conda env workaround for analysis3-26.03+; no-op on healthy envs
from exptdata_access import guess_experiment_from_esm_file
expt_key, info = guess_experiment_from_esm_file(esm_file)
model_name = info["model"]  # OM3 or CM3

In [ ]:
import os
plotfolder = str(cwd) + "/" if cwd else "./"
os.makedirs(plotfolder, exist_ok=True)
notebook_name="Bottom_age_tracer_in_ACCESS_OM3"

In [ ]:
import intake
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cmocean as cm
from distributed import Client

# Stop annoying warnings coming out of xarray.
import warnings
warnings.filterwarnings('ignore')

fig_path = plotfolder

In [ ]:
client = Client(threads_per_worker=1) 
from model_agnostic import patch_dask_workers
patch_dask_workers(client)  # patch workers too
print(client.dashboard_link)

Open intake datastore:

In [ ]:
datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=[
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
    ]
)

## Select year to plot

In [ ]:
# Define year to open, chose one of the examples or change manually
dates = '1981.*' if model_name == "ACCESS-CM3" else '1970.*'  # early year with full annual coverage in each run
# dates = '2020.*'  # Assumes IAF simulation, later in the run

User should not need to adjust anything in code below.

## Load OM3-25km bottom age tracer

In [ ]:
age_OM3 = datastore.search(variable="agessc", frequency="1mon", start_date=dates).to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=True
    )
)

In [ ]:
age_OM3

In [ ]:
# Refine area and take annual mean
age_OM3 = age_OM3.agessc.sel(yh=slice(-90,-50)).mean('time').load() 

# Get bottom tracer
depth_array_OM3 = age_OM3 * 0 + age_OM3.z_l
max_depth_OM3   = depth_array_OM3.max(dim='z_l', skipna=True)

age_bot_OM3 = age_OM3.where(depth_array_OM3.z_l >= max_depth_OM3)
age_bot_OM3 = age_bot_OM3.sum(dim='z_l').load()

# Finally, normalise age tracer by the max value
age_bot_normalised_OM3 = age_bot_OM3/age_bot_OM3.max()

#### Mask contiental shelf

Instructions for how to extract the shelf mask for the 1000 m isobath in OM3-25km can be found in the COSIMA recipes under [Create_contour.ipynb](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Create_contour.ipynb).

In [ ]:
# Load shelf mask (1000 m isobath on the matching OM3 grid)
path_OM3_1000m = {
    '25km': '/g/data/ol01/outputs/access-om3-25km/Antarctic_slope_contours/Antarctic_slope_contour_1000m_OM3_25km.nc',
    '100km': '/g/data/ol01/outputs/access-om3-100km/Antarctic_slope_contours/Antarctic_slope_contour_1000m_OM3_100km.nc',
}
om3_res = '100km' if age_OM3.sizes['xh'] == 360 else '25km'

contour_masked_above_OM3 = xr.open_mfdataset(path_OM3_1000m[om3_res]).contour_masked_above

shelf_mask_OM3 = contour_masked_above_OM3.where(contour_masked_above_OM3==0, 1)
shelf_mask_OM3 = shelf_mask_OM3.where(shelf_mask_OM3==1)

In [ ]:
# Mask the data
age_bot_masked_OM3 = age_bot_normalised_OM3*shelf_mask_OM3

## Compare to age tracer from OM2-01

Note there is no age tracer saved for OM2-025 (only for the first three years and then from 2300 onwards). OM2-01 has good DSW overflow and AABW formation so it makes sense to compare to this configuration.

In [ ]:
catalog  = intake.cat.access_nri
expt_01 = '01deg_jra55v140_iaf' # cylce 1 of OM2-01 IAF simulation
# expt_01  = '01deg_jra55v13_ryf9091' # use for RYF comparison
# expt_025 = '025deg_jra55_ryf9091_gadi'

In [ ]:
age_OM2_01 = catalog[expt_01].search(variable="age_global", frequency="1mon", start_date=dates).to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=True
    )
)

In [ ]:
# Refine area and take annual mean
age_OM2_01 = age_OM2_01.age_global.sel(yt_ocean=slice(-90,-50)).mean('time').load() 

# Get bottom tracer
depth_array_OM2_01 = age_OM2_01 * 0 + age_OM2_01.st_ocean
max_depth_OM2_01   = depth_array_OM2_01.max(dim='st_ocean', skipna=True)

age_bot_OM2_01 = age_OM2_01.where(depth_array_OM2_01.st_ocean >= max_depth_OM2_01)
age_bot_OM2_01 = age_bot_OM2_01.sum(dim='st_ocean').load()

# Finally, normalise age tracer by the max value
age_bot_normalised_OM2_01 = age_bot_OM2_01/age_bot_OM2_01.max()

#### Mask continental shelf

The file for the OM2-01 grid is saved as a *.npz file. Let's convert it to a dataarray for easier handling.

In [ ]:
path_OM2_01_1000m = '/g/data/ik11/grids/Antarctic_slope_contour_1000m.npz'  # original /g/data/g40/akm157/... path no longer exists; identical file lives here

contour_data_OM2_01  = np.load(path_OM2_01_1000m)

# Extract relevant information and write as dataarray
contour_masked_above_OM2_01 = xr.DataArray(
    contour_data_OM2_01['contour_masked_above'],
    dims=("yt_ocean", "xt_ocean"),
    coords={
        "xt_ocean": contour_data_OM2_01['xt_ocean'],
        "yt_ocean": contour_data_OM2_01['yt_ocean'],
    }
)

shelf_mask_OM2_01 = contour_masked_above_OM2_01.where(contour_masked_above_OM2_01==0, 1)
shelf_mask_OM2_01 = shelf_mask_OM2_01.where(shelf_mask_OM2_01==1)

In [ ]:
# Mask the data
age_bot_masked_OM2_01 = age_bot_normalised_OM2_01*shelf_mask_OM2_01

## Plotting

In [ ]:
# Land mask
var_search = datastore.search(variable="deptho")
ds = var_search.search(path=var_search.df.path[0]).to_dask() # we only need one file as bathymetry doesn't change
bathymetry = ds["deptho"].load()
land = xr.where(np.isnan(bathymetry.rename("land")), 1, np.nan)
land = land.sel(yh=slice(-90,-50))

ft_size = 12 # fontsize

In [ ]:
# 1000 m contour
slope_contour = contour_masked_above_OM3.where(contour_masked_above_OM3>0)
slope_contour = slope_contour/slope_contour

Plot MC_25km_jra_iaf-1.0-beta-5165c0f8 and 01deg_jra55v140_iaf (year 2014)

In [ ]:
plt.figure(figsize=(10, 6))

plt.subplot(211)
land.plot.contourf(colors="darkgrey", add_colorbar=False)
age_bot_masked_OM3.plot(cmap=cm.cm.rain_r, vmin=0.5, vmax=1, cbar_kwargs={'label': "Normalised bottom age"})
slope_contour.fillna(2).plot.contour(levels=[0,1], colors='black', linewidths=1.5)
plt.title(f'ACCESS-OM3-{om3_res}'), plt.title('a', fontweight='bold', fontsize=ft_size, loc='left')
plt.tick_params(labelbottom=False)
plt.xlabel(''), plt.ylabel('Latitude (°N)', fontsize=ft_size)
plt.text(-275, -77, 'Year '+dates[0:4], fontsize=ft_size)
plt.text(-275, -80, get_exptname_from_path(esm_file), fontsize=ft_size)

plt.subplot(212)
land.plot.contourf(colors="darkgrey", add_colorbar=False)
age_bot_masked_OM2_01.plot(cmap=cm.cm.rain_r, vmin=0.5, vmax=1, cbar_kwargs={'label': "Normalised bottom age"})
slope_contour.fillna(2).plot.contour(levels=[0,1], colors='black', linewidths=1.5)
plt.title('ACCESS-OM2-01'), plt.title('b', fontweight='bold', fontsize=ft_size, loc='left')
plt.xlabel('Longitude (°E)', fontsize=ft_size), plt.ylabel('Latitude (°N)', fontsize=ft_size)
plt.text(-275, -77, 'Year '+dates[0:4], fontsize=ft_size)
plt.text(-275, -80, '01deg_jra55v140_iaf', fontsize=ft_size)

mkmd.savefig(plt.gcf(),"Bottom age tracer", "Normalised bottom age comparing ACCESS-OM3 and ACCESS-OM2 at year "+dates[0:4] , dpi=150)